# Prédiction de densité de foule — LSTM vs régression linéaire
**Projet Smart Crowd AI · JOJ Dakar 2026**

Objectif : comparer un modèle LSTM à la baseline régression linéaire actuellement
déployée dans `server.py` pour prédire la densité au prochain pas de temps.

| | |
|---|---|
| **Données** | 115 200 points · 80 zones · 30 jours · intervalle 30 min |
| **Séquences** | 10 points d'entrée → prédire le suivant |
| **Métriques** | MAE et RMSE en points de densité (%) |
| **Export** | Poids numpy → inférence sans TensorFlow sur Render |

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import timedelta
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.random.set_seed(42)
np.random.seed(42)

# Résolution robuste des chemins (notebook/ ou racine)
_cwd = os.getcwd()
ROOT = _cwd if os.path.exists(os.path.join(_cwd, 'server.py')) else os.path.dirname(_cwd)
CSV_PATH   = os.path.join(ROOT, 'data',  'historique_simule.csv')
MODEL_DIR  = os.path.join(ROOT, 'core',  'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'TensorFlow {tf.__version__}')
print(f'ROOT : {ROOT}')

## 1. Chargement des données

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=['timestamp'])
df = df.sort_values(['lieu', 'zone', 'timestamp']).reset_index(drop=True)

n_pairs = df.groupby(['lieu', 'zone']).ngroups
print(f'Lignes            : {len(df):,}')
print(f'Paires (lieu,zone): {n_pairs}')
print(f'Période           : {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'Densité           : min={df["densite"].min()}%  max={df["densite"].max()}%  moy={df["densite"].mean():.1f}%')
df.head(3)

## 2. EDA — Distributions et saisonnalité

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#060D12')
for ax in axes:
    ax.set_facecolor('#0D1A21')
    ax.tick_params(colors='#ECF4F8')
    ax.xaxis.label.set_color('#ECF4F8')
    ax.yaxis.label.set_color('#ECF4F8')
    ax.title.set_color('#ECF4F8')
    for spine in ax.spines.values(): spine.set_edgecolor('#182832')

# Distribution globale
axes[0].hist(df['densite'], bins=40, color='#00F0A0', edgecolor='#060D12', linewidth=0.4)
axes[0].set_title('Distribution globale')
axes[0].set_xlabel('Densité (%)')
axes[0].set_ylabel('Fréquence')

# Jour normal vs événement
ev = df[df['is_event_day'] == 1]['densite']
no = df[df['is_event_day'] == 0]['densite']
axes[1].hist(no, bins=40, alpha=0.65, label=f'Normal  moy={no.mean():.1f}%', color='#4A7080')
axes[1].hist(ev, bins=40, alpha=0.65, label=f'Événement moy={ev.mean():.1f}%', color='#FF3D3D')
axes[1].set_title('Normal vs Événement')
axes[1].set_xlabel('Densité (%)')
axes[1].legend(fontsize=8, facecolor='#112330', labelcolor='#ECF4F8')

# Profil horaire moyen
hourly = df.groupby(df['timestamp'].dt.hour)['densite'].mean()
axes[2].plot(hourly.index, hourly.values, color='#FFBA00', linewidth=2, marker='o', markersize=4)
axes[2].set_title('Profil horaire moyen')
axes[2].set_xlabel('Heure')
axes[2].set_ylabel('Densité moy. (%)')
axes[2].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'eda_distributions.png'), dpi=120, bbox_inches='tight',
            facecolor='#060D12')
plt.show()
print('Pics à 9h, 14h, 18-19h — cohérents avec les créneaux de compétition JOJ.')

In [ ]:
# Série temporelle sur 14 jours — zone représentative
ZONE_REF = ('Stade Iba Mar Diop', 'Entree Principale')
sample = df[(df['lieu'] == ZONE_REF[0]) & (df['zone'] == ZONE_REF[1])].copy()
sample = sample[sample['timestamp'] < sample['timestamp'].min() + timedelta(days=14)]

fig, ax = plt.subplots(figsize=(14, 4), facecolor='#060D12')
ax.set_facecolor('#0D1A21')
ax.plot(sample['timestamp'], sample['densite'], color='#00F0A0', linewidth=0.8)
# Surligne les jours événement
for day_idx, grp in sample.groupby(sample['timestamp'].dt.date):
    if grp['is_event_day'].iloc[0] == 1:
        ax.axvspan(grp['timestamp'].iloc[0], grp['timestamp'].iloc[-1],
                   color='#FF3D3D', alpha=0.08)
ax.set_title(f'{ZONE_REF[0]} — {ZONE_REF[1]} (14 premiers jours)',
             color='#ECF4F8')
ax.set_xlabel('Date', color='#ECF4F8')
ax.set_ylabel('Densité (%)', color='#ECF4F8')
ax.tick_params(colors='#ECF4F8')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
for spine in ax.spines.values(): spine.set_edgecolor('#182832')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'eda_serie_temporelle.png'), dpi=120,
            bbox_inches='tight', facecolor='#060D12')
plt.show()
print('Zone en rouge = jour événement (J+7). Pic visible sur la série.')

## 3. Préparation des séquences

In [ ]:
SEQ_LEN     = 10    # 10 points = 5h d'historique
TRAIN_RATIO = 0.80

# Normalisation globale [0, 1] — plage connue [5, 100]
D_MIN, D_MAX = 5.0, 100.0
def normalize(x):   return (np.asarray(x) - D_MIN) / (D_MAX - D_MIN)
def denormalize(x): return np.asarray(x) * (D_MAX - D_MIN) + D_MIN

X_train, y_train = [], []
X_test,  y_test  = [], []

for (lieu, zone), grp in df.groupby(['lieu', 'zone']):
    series  = normalize(grp.sort_values('timestamp')['densite'].values)
    n_train = int(len(series) * TRAIN_RATIO)
    for i in range(SEQ_LEN, n_train):
        X_train.append(series[i - SEQ_LEN:i])
        y_train.append(series[i])
    for i in range(max(SEQ_LEN, n_train), len(series)):
        X_test.append(series[i - SEQ_LEN:i])
        y_test.append(series[i])

X_train = np.array(X_train, dtype=np.float32).reshape(-1, SEQ_LEN, 1)
y_train = np.array(y_train, dtype=np.float32)
X_test  = np.array(X_test,  dtype=np.float32).reshape(-1, SEQ_LEN, 1)
y_test  = np.array(y_test,  dtype=np.float32)

print(f'Train : {X_train.shape[0]:,} séquences  shape={X_train.shape}')
print(f'Test  : {X_test.shape[0]:,}  séquences  shape={X_test.shape}')

## 4. Baseline — régression linéaire (logique identique à `server.py`)

In [ ]:
def linear_regression_predict(seq):
    """Régression linéaire sur SEQ_LEN points → extrapole le suivant.
    Même implémentation que ZONE_HISTORY dans /api/refresh de server.py."""
    n      = len(seq)
    x_mean = (n - 1) / 2
    y_mean = seq.mean()
    num    = sum((j - x_mean) * (seq[j] - y_mean) for j in range(n))
    den    = sum((j - x_mean) ** 2 for j in range(n)) or 1
    slope  = num / den
    return float(np.clip(seq[-1] + slope, 0, 1))

y_pred_base = np.array([
    linear_regression_predict(X_test[i, :, 0]) for i in range(len(X_test))
], dtype=np.float32)

SCALE = D_MAX - D_MIN  # 95 — facteur pour convertir erreur normalisée en %

mae_base  = mean_absolute_error(y_test, y_pred_base) * SCALE
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base)) * SCALE
print(f'Baseline régression linéaire — MAE : {mae_base:.2f}%   RMSE : {rmse_base:.2f}%')

## 5. Modèle LSTM

In [ ]:
UNITS = 32

model = Sequential([
    LSTM(UNITS, input_shape=(SEQ_LEN, 1)),
    Dense(1)
], name='SmartCrowdLSTM')
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5,
                           restore_best_weights=True, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=512,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), facecolor='#060D12')
ax.set_facecolor('#0D1A21')
ax.plot(history.history['loss'],     label='Train loss', color='#00F0A0', linewidth=2)
ax.plot(history.history['val_loss'], label='Val loss',   color='#FFBA00', linewidth=2)
ax.set_title("Courbe d'apprentissage", color='#ECF4F8')
ax.set_xlabel('Époque', color='#ECF4F8')
ax.set_ylabel('MSE (espace normalisé)', color='#ECF4F8')
ax.tick_params(colors='#ECF4F8')
ax.legend(facecolor='#112330', labelcolor='#ECF4F8')
for spine in ax.spines.values(): spine.set_edgecolor('#182832')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_curve.png'), dpi=120,
            bbox_inches='tight', facecolor='#060D12')
plt.show()

## 6. Évaluation comparative

In [ ]:
y_pred_lstm = model.predict(X_test, batch_size=512, verbose=0).flatten()

mae_lstm  = mean_absolute_error(y_test, y_pred_lstm) * SCALE
rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm)) * SCALE

gain_mae  = (mae_base  - mae_lstm)  / mae_base  * 100
gain_rmse = (rmse_base - rmse_lstm) / rmse_base * 100

print(f'{"":─<48}')
print(f'{"Modèle":<26} {"MAE (%)":>9} {"RMSE (%)":>9}')
print(f'{"":─<48}')
print(f'{"Régression linéaire (baseline)":<26} {mae_base:>9.2f} {rmse_base:>9.2f}')
print(f'{"LSTM 32 units":<26} {mae_lstm:>9.2f} {rmse_lstm:>9.2f}')
print(f'{"":─<48}')
print(f'Gain LSTM   MAE : {gain_mae:+.1f}%    RMSE : {gain_rmse:+.1f}%')

## 7. Prédiction vs réel

In [ ]:
# Série de référence pour visualisation — 2 derniers jours (96 pts)
ref = df[(df['lieu'] == ZONE_REF[0]) & (df['zone'] == ZONE_REF[1])]
ref_series = normalize(ref.sort_values('timestamp')['densite'].values)

N_VIS = 96
vis   = ref_series[-(N_VIS + SEQ_LEN):]

y_real, y_lstm_vis, y_base_vis = [], [], []
for i in range(SEQ_LEN, len(vis)):
    seq = vis[i - SEQ_LEN:i].astype(np.float32)
    y_real.append(vis[i])
    y_lstm_vis.append(model.predict(seq.reshape(1, SEQ_LEN, 1), verbose=0)[0, 0])
    y_base_vis.append(linear_regression_predict(seq))

y_real     = denormalize(np.array(y_real))
y_lstm_vis = denormalize(np.array(y_lstm_vis))
y_base_vis = denormalize(np.array(y_base_vis))

fig, ax = plt.subplots(figsize=(14, 5), facecolor='#060D12')
ax.set_facecolor('#0D1A21')
t = range(len(y_real))
ax.plot(t, y_real,     label='Réel',                color='#ECF4F8', linewidth=1.5)
ax.plot(t, y_lstm_vis, label='LSTM',                color='#00F0A0', linewidth=1.5, linestyle='--')
ax.plot(t, y_base_vis, label='Régression linéaire', color='#FFBA00', linewidth=1.0, linestyle=':')
ax.fill_between(t, y_real, y_lstm_vis, alpha=0.07, color='#00F0A0')
ax.set_title(f"{ZONE_REF[0]} — {ZONE_REF[1]} · 96 derniers points (48h)",
             color='#ECF4F8')
ax.set_xlabel('Pas de temps (×30 min)', color='#ECF4F8')
ax.set_ylabel('Densité (%)', color='#ECF4F8')
ax.set_ylim(0, 105)
ax.tick_params(colors='#ECF4F8')
ax.legend(facecolor='#112330', labelcolor='#ECF4F8')
for spine in ax.spines.values(): spine.set_edgecolor('#182832')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'prediction_vs_reel.png'), dpi=120,
            bbox_inches='tight', facecolor='#060D12')
plt.show()

## 8. Export du modèle

In [ ]:
# Sauvegarde Keras complète (re-entraînement / référence)
model.save(os.path.join(MODEL_DIR, 'lstm_crowd.keras'))

# Export poids numpy — inférence sans TF en production
# Structure weights[] pour LSTM(32) + Dense(1) :
#   [0] lstm_kernel           (1, 128)   — projections input → 4 gates
#   [1] lstm_recurrent_kernel (32, 128)  — projections h_{t-1} → 4 gates
#   [2] lstm_bias             (128,)     — biais des 4 gates
#   [3] dense_kernel          (32, 1)    — projection finale
#   [4] dense_bias            (1,)       — biais dense
weights = model.get_weights()
names   = ['lstm_kernel', 'lstm_recurrent_kernel', 'lstm_bias',
           'dense_kernel', 'dense_bias']
for name, w in zip(names, weights):
    path = os.path.join(MODEL_DIR, f'{name}.npy')
    np.save(path, w)
    print(f'  {name}.npy  shape={w.shape}')

# Paramètres de normalisation
scaler = {'d_min': D_MIN, 'd_max': D_MAX, 'units': UNITS, 'seq_len': SEQ_LEN}
with open(os.path.join(MODEL_DIR, 'scaler.json'), 'w') as f:
    json.dump(scaler, f, indent=2)

print('\nscaler.json :', scaler)
print('\nModèle Keras : lstm_crowd.keras')

## Conclusion

| Modèle | MAE (%) | RMSE (%) |
|---|---|---|
| Régression linéaire (baseline) | voir ci-dessus | voir ci-dessus |
| LSTM 32 units | voir ci-dessus | voir ci-dessus |

### Interprétation honnête

Le gain du LSTM est **réel mais modeste** sur données simulées. C'est attendu :

- Les séries sont générées avec du **bruit gaussien** (σ=4%) — aucun modèle ne peut
  prédire le bruit. La limite basse théorique de MAE ≈ σ × √(2/π) ≈ **3.2%**.
- La régression linéaire est une baseline **déjà forte** sur des séries lisses avec
  tendance locale.
- Le LSTM apporte un gain sur les **transitions non-linéaires** : bascule jour/nuit,
  pics d'événement, plateaux — zones où la pente locale est un mauvais prédicteur.

### En production réelle (données Orange Network Analytics)

Le gain serait plus marqué car les séries réelles contiennent :
- des **patterns non-stochastiques** (habitudes de foule, effets transport)
- des **corrélations entre zones** exploitables par un modèle multi-varié
- des **événements imprévus** que la mémoire LSTM gère mieux qu'une pente

### Intégration

Les poids numpy dans `core/models/` permettent une inférence dans `core/prediction.py`
**sans TensorFlow** sur Render (zéro nouvelle dépendance production).